In [1]:

import os
import csv
import hashlib
import requests
from datetime import datetime, timezone
from pathlib import Path

print("Data ingestion libraries imported successfully!")


Data ingestion libraries imported successfully!


In [2]:

# Cell 2: Define dataset sources and local storage folder

DATASET_NAME = "Building Data Genome Project 2"

BASE_URL = (
    "https://media.githubusercontent.com/media/"
    "buds-lab/building-data-genome-project-2/"
    "refs/heads/master/data"
)

DATASET_URLS = {
    "electricity.csv": BASE_URL + "/meters/raw/electricity.csv",
    "metadata.csv": BASE_URL + "/metadata/metadata.csv"
}

# Create a local directory in the Colab runtime
DATA_DIR = Path("/content/building_energy_raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_NAME)
print("Local folder:", DATA_DIR)

for filename, url in DATASET_URLS.items():
    print(filename, "->", url)


Dataset: Building Data Genome Project 2
Local folder: /content/building_energy_raw
electricity.csv -> https://media.githubusercontent.com/media/buds-lab/building-data-genome-project-2/refs/heads/master/data/meters/raw/electricity.csv
metadata.csv -> https://media.githubusercontent.com/media/buds-lab/building-data-genome-project-2/refs/heads/master/data/metadata/metadata.csv


In [3]:

# Cell 3: Download original datasets from GitHub

for filename, url in DATASET_URLS.items():
    file_path = DATA_DIR / filename

    print(f"Downloading {filename}...")

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    file_size_mb = file_path.stat().st_size / (1024 * 1024)

    print(f"Downloaded: {filename}")
    print(f"Size: {file_size_mb:.2f} MB")
    print("-" * 40)

print("All original datasets downloaded successfully!")

Downloaded: electricity.csv
Size: 166.17 MB
----------------------------------------
Downloaded: metadata.csv
Size: 0.26 MB
----------------------------------------
All original datasets downloaded successfully!


In [4]:

# Cell 4: Verify the downloaded CSV files

for filename in DATASET_URLS:
    file_path = DATA_DIR / filename

    print(f"Checking: {filename}")

    # Check if the file exists
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {filename}")

    # Check the file size
    file_size = file_path.stat().st_size

    if file_size == 0:
        raise ValueError(f"Empty file: {filename}")

    # Read only the CSV header
    with open(file_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        columns = next(reader)

    print("File exists: Yes")
    print(f"File size: {file_size:,} bytes")
    print(f"Number of columns: {len(columns)}")
    print(f"First 5 columns: {columns[:5]}")
    print("Verification: PASSED")
    print("-" * 45)

print("All files passed basic verification!")

Checking: electricity.csv
File exists: Yes
File size: 174,239,039 bytes
Number of columns: 1579
First 5 columns: ['timestamp', 'Panther_parking_Lorriane', 'Panther_lodging_Cora', 'Panther_office_Hannah', 'Panther_lodging_Hattie']
Verification: PASSED
---------------------------------------------
Checking: metadata.csv
File exists: Yes
File size: 272,024 bytes
Number of columns: 32
First 5 columns: ['building_id', 'site_id', 'building_id_kaggle', 'site_id_kaggle', 'primaryspaceusage']
Verification: PASSED
---------------------------------------------
All files passed basic verification!


In [5]:

# Cell 5: Create data-ingestion log

ingestion_time = datetime.now(timezone.utc).isoformat()

log_records = []

for filename, url in DATASET_URLS.items():
    file_path = DATA_DIR / filename

    # Calculate SHA-256 checksum
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    # Create one log entry for each file
    record = {
        "dataset": DATASET_NAME,
        "filename": filename,
        "source_url": url,
        "ingestion_time_utc": ingestion_time,
        "file_size_bytes": file_path.stat().st_size,
        "sha256": sha256.hexdigest(),
        "verification_status": "PASSED",
        "local_path": str(file_path)
    }

    log_records.append(record)

# Save ingestion log as CSV
LOG_PATH = DATA_DIR / "ingestion_log.csv"

with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=log_records[0].keys()
    )
    writer.writeheader()
    writer.writerows(log_records)

print("Ingestion log created successfully!")
print("Saved at:", LOG_PATH)

for record in log_records:
    print(
        record["filename"],
        "| Size:", record["file_size_bytes"],
        "| Status:", record["verification_status"]
    )

Ingestion log created successfully!
Saved at: /content/building_energy_raw/ingestion_log.csv
electricity.csv | Size: 174239039 | Status: PASSED
metadata.csv | Size: 272024 | Status: PASSED


In [ ]:
from google.colab import drive
drive.mount('/content/drive')